# Guardian Pixel — Layer A Training

This notebook trains the first detection layer of the Guardian Pixel system.

**Architecture:** ConvNeXt-Large backbone (ImageNet pretrained) with Filter Stride Reduction (FSR), fine-tuned for 7-class fake-image detection.  
**External output:** Binary (real vs. fake). The 7-class head is used internally to improve robustness and representation.  
**Data:** ArtiFact dataset, accessed via `master_metadata.csv` — images are fetched directly by path, no folder traversal.

---
### Notebook flow
1. Mount Google Drive
2. Unzip ArtiFact dataset to the Colab session disk
3. Clone the repository and install dependencies
4. Verify GPU and run model sanity check
5. Train the model
6. Evaluate on the test set with full metrics

## 1. Mount Google Drive

The dataset zip and `master_metadata.csv` both live on Google Drive.  
Mounting first ensures both are accessible before anything else runs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH      = '/content/drive/MyDrive/Kaggle_Datasets/master_metadata.csv'
ZIP_PATH      = '/content/drive/MyDrive/Kaggle_Datasets/artifact-dataset.zip'
ARTIFACT_ROOT = '/content/ArtiFact'

print(f'CSV path      : {CSV_PATH}')
print(f'Zip path      : {ZIP_PATH}')
print(f'Artifact root : {ARTIFACT_ROOT}')

## 2. Unzip ArtiFact to Session Disk

The zip stays on Drive permanently.  Each session we extract it to the local runtime disk (`/content/ArtiFact`) for fast image reads during training — Drive I/O would be a bottleneck with 2.5 M images.

The cell is skipped automatically if the folder already exists (e.g. if the runtime was reconnected).

In [ ]:
import os, zipfile

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        f'Zip not found: {ZIP_PATH}\n'
        'Update ZIP_PATH in Cell 1 to match your Drive folder structure.'
    )

if os.path.isdir(ARTIFACT_ROOT):
    print(f'ArtiFact already unzipped this session — skipping.')
else:
    print('Counting files in zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        total = len(zf.namelist())
    print(f'Extracting {total:,} files to {ARTIFACT_ROOT} ...')
    os.makedirs(ARTIFACT_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(ARTIFACT_ROOT)
    print('Done.')

print(f'\nARTIFACT_ROOT = {ARTIFACT_ROOT}')

## 3. Clone Repository & Install Dependencies

The model code lives in the `guardian-pixel` GitHub repository.  
We clone it (or pull the latest changes if already cloned) and add it to the Python path so all modules are importable.

Dependencies not pre-installed in Colab are installed quietly.

In [ ]:
import os, sys

REPO_DIR = '/content/guardian-pixel'

if os.path.isdir(REPO_DIR):
    print('Repo already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/GorohovskiMax/guardian-pixel.git {REPO_DIR}

%cd /content/guardian-pixel
sys.path.insert(0, '/content/guardian-pixel')
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q timm albumentations wandb scikit-learn tqdm
print('Dependencies ready.')

## 4. Weights & Biases Login

W&B is used to log training metrics (loss, accuracy, learning rate) and validation scores after each epoch.  
Paste your API key when prompted — get it at https://wandb.ai/authorize.  
Training will continue without W&B if login fails.

In [ ]:
import wandb
wandb.login()

## 5. Verify GPU

Training requires a GPU.  This cell confirms CUDA is available and prints device info.  
If no GPU is detected, go to **Runtime → Change runtime type → GPU** before proceeding.

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU     : {props.name}')
    print(f'VRAM    : {props.total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be extremely slow.')

!nvidia-smi

## 6. Model Sanity Check

Before training, we verify two things:

1. **FSR applied** — the stem `Conv2d` stride should be `(2, 2)`, not `(4, 4)`.
2. **Forward pass shape** — a dummy batch must produce logits of shape `(1, 7)` for the 7-class head.

If either assertion fails, something is wrong with the model definition — do not proceed to training.

In [ ]:
import torch
from layers.forensic import ForensicDetector

CONFIG = 'configs/layer_a.yaml'

model = ForensicDetector.from_config(CONFIG)

print('=== Stem after FSR ===')
print(model.model.stem[0])
assert model.model.stem[0].stride == (2, 2), 'FSR not applied — stem stride is not (2, 2)'
print('FSR: OK\n')

dummy = torch.randn(1, 3, 200, 200)
model.eval()
with torch.no_grad():
    logits = model(dummy)

assert logits.shape == (1, 7), f'Expected (1, 7), got {tuple(logits.shape)}'
print(f'Forward pass output shape : {tuple(logits.shape)}  OK')

## 7. Train

The training loop:
- Loads images from `master_metadata.csv` where `split == train`
- Each image path is resolved as `ARTIFACT_ROOT / image_path` — no folder traversal
- After every epoch, the model is evaluated on `split == validation`
- The best checkpoint (highest validation balanced accuracy) is saved to `models/layer_a/best.pt`
- All hyperparameters are read from `configs/layer_a.yaml`

**Balanced accuracy** is the primary validation metric because it accounts for class imbalance across the 7 classes.

In [ ]:
from layers.training import train

trained_model = train(
    config_path   = 'configs/layer_a.yaml',
    csv_path      = CSV_PATH,
    artifact_root = ARTIFACT_ROOT,
    wandb_project = 'guardian-pixel',
)

## 8. Test Set Evaluation

The best checkpoint is loaded and evaluated on the held-out test set (`split == test`).  
The test set was never seen during training or used for checkpoint selection.

**Metrics reported:**

| Metric | Description |
|--------|-------------|
| Loss | Cross-entropy loss |
| Accuracy | Overall 7-class accuracy |
| Balanced Accuracy | Macro-average per-class recall — robust to imbalance |
| Per-class P/R/F1 | Precision, recall, F1 for each of the 7 classes |
| Confusion Matrix | 7×7 matrix of predicted vs. true labels |
| Binary Accuracy | Accuracy after collapsing to Real vs. Fake |
| Binary Precision | Of all images predicted fake, how many were actually fake |
| Binary Recall | Of all fake images, how many were correctly detected |
| Binary F1 | Harmonic mean of binary precision and recall |
| ROC-AUC | Area under the ROC curve for the binary Real vs. Fake decision |

In [ ]:
import torch
import numpy as np
from layers.forensic import ForensicDetector
from layers.training import evaluate
from utils.dataloader import get_dataloaders
from utils.dataset import CLASS_NAMES

CONFIG     = 'configs/layer_a.yaml'
CHECKPOINT = 'models/layer_a/best.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load best checkpoint
model = ForensicDetector.from_config(CONFIG).to(device)
checkpoint = torch.load(CHECKPOINT, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Checkpoint loaded — epoch {checkpoint["epoch"]},  '
      f'val_bal_acc = {checkpoint["best_balanced_accuracy"]:.4f}\n')

# Build test loader
loaders = get_dataloaders(CONFIG, csv_path=CSV_PATH, artifact_root=ARTIFACT_ROOT)

# Evaluate
metrics = evaluate(model, loaders['test'], device)

# ---- 7-class results ----
print('=' * 55)
print('  7-CLASS RESULTS')
print('=' * 55)
print(f'  Loss                : {metrics["loss"]:.4f}')
print(f'  Accuracy            : {metrics["accuracy"]:.4f}')
print(f'  Balanced Accuracy   : {metrics["balanced_accuracy"]:.4f}')

print('\n  Per-class breakdown:')
report = metrics['per_class_report']
header = f'  {"Class":<22} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}'
print(header)
print('  ' + '-' * 66)
for cls in CLASS_NAMES:
    if cls in report:
        r = report[cls]
        print(f'  {cls:<22} {r["precision"]:>10.4f} {r["recall"]:>10.4f} '
              f'{r["f1-score"]:>10.4f} {int(r["support"]):>10,}')

print('\n  Confusion Matrix (rows=true, cols=predicted):')
cm = metrics['confusion_matrix']
label_str = '  ' + ''.join(f'{i:>8}' for i in range(len(CLASS_NAMES)))
print(label_str)
for i, row in enumerate(cm):
    print(f'  {i}' + ''.join(f'{v:>8,}' for v in row))

# ---- Binary results ----
print()
print('=' * 55)
print('  BINARY RESULTS  (Real vs. Fake)')
print('=' * 55)
print(f'  Accuracy            : {metrics["binary_accuracy"]:.4f}')
print(f'  Precision           : {metrics["binary_precision"]:.4f}')
print(f'  Recall              : {metrics["binary_recall"]:.4f}')
print(f'  F1                  : {metrics["binary_f1"]:.4f}')
print(f'  ROC-AUC             : {metrics["binary_roc_auc"]:.4f}')